# 05. OpenaiAdapter — deep dive

`OpenaiAdapter` is the translation layer between ArcLLM's universal
types (`Message`, `Tool`, `LLMResponse`) and the OpenAI Chat
Completions API. It is the second adapter most callers reach for —
and the first one that introduces a real wire-format shift away from
Anthropic: tool calls move from content blocks to `message.tool_calls`,
tool arguments are JSON strings rather than dicts, tool results need
1-to-N flattening, system-role messages get rewritten to `developer`
for the o-series, and `max_tokens` becomes `max_completion_tokens` for
reasoning models.

This notebook is the single-adapter deep dive. The catalog and
cross-provider comparison live in `arcllm/13-open-providers`. The
Anthropic-side equivalent lives in `arcllm/03-anthropic-adapter`. We
do not duplicate that material here.

You will learn:
- How `OpenaiAdapter` extends `BaseAdapter` and what each method does
- How a `list[Message]` becomes a fully-formed Chat Completions body
- How `_format_messages` flattens one ArcLLM tool message into N OpenAI ones
- How o-series detection rewrites `system` → `developer` and swaps the budget keys
- How `_STOP_REASON_MAP` normalizes OpenAI `finish_reason` values
- How tool calls round-trip end-to-end with a mocked HTTP client
- How streaming chunk shapes look on the wire (illustrative)
- How errors surface as `ArcLLMAPIError`, including `Retry-After` parsing

## 1. Setup

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

The live API section near the bottom is gated by `HAS_LIVE_KEY`.
Every other cell runs deterministically against a patched
`httpx.AsyncClient.post` — the entire notebook works without an
internet connection or an OpenAI key.

In [2]:
# (live) optional — set this to run real-API sections; mock cells run regardless
HAS_LIVE_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print(f"Live API key: {'present' if HAS_LIVE_KEY else 'missing — live cells will skip'}")

Live API key: present


### Imports and a fixture config

Two fixture configs — one standard (`gpt-4.1`, `supports_thinking=False`),
one o-series (`o4-mini`, `supports_thinking=True`). The standard fixture
exercises the chat-completions path; the o-series fixture exercises the
reasoning-model path (developer role, `max_completion_tokens`,
`reasoning_effort`).

In [3]:
import json
from typing import Any
from unittest.mock import AsyncMock, patch

import httpx

from arcllm import (
    ArcLLMAPIError,
    ArcLLMConfigError,
    ArcLLMParseError,
    BaseAdapter,
    ImageBlock,
    LLMResponse,
    Message,
    ModelMetadata,
    OpenaiAdapter,
    ProviderConfig,
    ProviderSettings,
    StopReason,
    TextBlock,
    Tool,
    ToolCall,
    ToolResultBlock,
    ToolUseBlock,
    Usage,
    load_provider_config,
)

# Test API key — never read by OpenAI, only by the adapter at init.
os.environ["ARCLLM_TEST_KEY"] = "sk-test-walkthrough-key"

STANDARD_MODEL = "gpt-4.1"
REASONING_MODEL = "o4-mini"

fake_config = ProviderConfig(
    provider=ProviderSettings(
        api_format="openai-chat",
        base_url="https://api.openai.com",
        api_key_env="ARCLLM_TEST_KEY",
        api_key_required=True,
        default_model=STANDARD_MODEL,
        default_temperature=0.7,
    ),
    models={
        STANDARD_MODEL: ModelMetadata(
            context_window=1_000_000,
            max_output_tokens=32_768,
            supports_tools=True,
            supports_vision=True,
            supports_thinking=False,
            input_modalities=["text", "image"],
            cost_input_per_1m=2.00,
            cost_output_per_1m=8.00,
            cost_cache_read_per_1m=0.50,
            cost_cache_write_per_1m=2.00,
        ),
        REASONING_MODEL: ModelMetadata(
            context_window=200_000,
            max_output_tokens=100_000,
            supports_tools=True,
            supports_vision=True,
            supports_thinking=True,  # o-series
            input_modalities=["text", "image"],
            cost_input_per_1m=1.10,
            cost_output_per_1m=4.40,
            cost_cache_read_per_1m=0.275,
            cost_cache_write_per_1m=1.10,
        ),
    },
)
print(f"default model: {fake_config.provider.default_model}")
print(f"models: {list(fake_config.models)}")

default model: gpt-4.1
models: ['gpt-4.1', 'o4-mini']


## 2. The adapter contract

`OpenaiAdapter` extends `BaseAdapter`, which itself implements the
abstract `LLMProvider` from `arcllm.types`. The contract every adapter
satisfies is small:

| Method | Purpose |
|---|---|
| `invoke(messages, tools, **kwargs)` | One round-trip, returns `LLMResponse` |
| `validate_config()` | True iff this adapter has the credentials it needs |
| `close()` | Release the underlying httpx client |

Everything else (`_build_headers`, `_format_message`, `_format_messages`,
`_format_tool`, `_build_request_body`, `_parse_response`,
`_parse_tool_call`, `_parse_usage`, `_map_stop_reason`,
`_is_reasoning_model`) is private plumbing — small, testable, single-purpose.

In [4]:
adapter = OpenaiAdapter(fake_config, STANDARD_MODEL)

print(f"adapter.name             = {adapter.name!r}")
print(f"adapter.model_name       = {adapter.model_name!r}")
print(f"isinstance BaseAdapter?  = {isinstance(adapter, BaseAdapter)}")
print(f"validate_config()        = {adapter.validate_config()}")
print(f"_is_reasoning_model      = {adapter._is_reasoning_model}")
print(f"_model_meta.supports_thinking = {adapter._model_meta.supports_thinking}")

adapter.name             = 'openai'
adapter.model_name       = 'gpt-4.1'
isinstance BaseAdapter?  = True
validate_config()        = True
_is_reasoning_model      = False
_model_meta.supports_thinking = False


### Fail-fast key resolution

`BaseAdapter.__init__` reads the env var named in `api_key_env`. If
the variable is missing or empty and `api_key_required=True`, it
raises `ArcLLMConfigError` immediately — not three hours into a batch
job.

In [5]:
# Missing key → ArcLLMConfigError at construction time
saved = os.environ.pop("ARCLLM_TEST_KEY")
try:
    OpenaiAdapter(fake_config, STANDARD_MODEL)
except ArcLLMConfigError as e:
    print(f"got: {type(e).__name__}: {e}")
finally:
    os.environ["ARCLLM_TEST_KEY"] = saved

got: ArcLLMConfigError: Missing environment variable 'ARCLLM_TEST_KEY' for provider. Set it to your API key.


In [6]:
# Vault-resolved key bypasses the env var entirely (NIST AU-9 / FedRAMP)
adapter_vault = OpenaiAdapter(
    fake_config,
    STANDARD_MODEL,
    resolved_api_key="sk-from-vault-not-env",
)
print(f"adapter_vault._api_key = {adapter_vault._api_key!r}")
print("(env var was bypassed — keeps secrets out of os.environ)")

adapter_vault._api_key = 'sk-from-vault-not-env'
(env var was bypassed — keeps secrets out of os.environ)


### Headers — Bearer token, not `x-api-key`

OpenAI authenticates with the standard `Authorization: Bearer …`
header. Anthropic uses `x-api-key`. The adapter caches the headers
dict at `__init__` because the API key is immutable for the
lifetime of the adapter.

In [7]:
headers = adapter._build_headers()
for k, v in headers.items():
    if k == "Authorization":
        v = v[:14] + "…"
    print(f"  {k}: {v}")

  Content-Type: application/json
  Authorization: Bearer sk-test…


## 3. Standard models — `gpt-4.1`, `gpt-4o`

Standard (non-reasoning) models honor the chat-completions defaults
you'd expect: `system` role stays `system`, `max_tokens` stays
`max_tokens`, `temperature` is sent as-is. This is the path 95% of
calls take.

### 3a. System messages stay in-line

Unlike Anthropic (which extracts `system` to a top-level field), OpenAI
keeps system messages in the `messages` array — `role="system"`. The
adapter just passes them through.

In [8]:
messages = [
    Message(role="system", content="You are a helpful assistant."),
    Message(role="user", content="Hello!"),
]
body = adapter._build_request_body(messages)
for m in body["messages"]:
    print(m)

{'role': 'system', 'content': 'You are a helpful assistant.'}
{'role': 'user', 'content': 'Hello!'}


### 3b. Multi-modal content blocks

Image inputs become `image_url` parts with a `data:` URL. Text-only
messages with a single string content pass through verbatim. The
adapter never wraps a plain string in a `[{type: text}]` array — that
would needlessly inflate token counts on the wire.

In [9]:
# String-content message: passes through as-is
msg = Message(role="user", content="Hello!")
print("string content:")
print(json.dumps(adapter._format_message(msg), indent=2))

# Image + text: becomes a parts array with image_url
msg = Message(
    role="user",
    content=[
        TextBlock(text="Describe this image:"),
        ImageBlock(source="iVBORw0KG…", media_type="image/png"),
    ],
)
print("\nmulti-part content:")
print(json.dumps(adapter._format_message(msg), indent=2))

string content:
{
  "role": "user",
  "content": "Hello!"
}

multi-part content:
{
  "role": "user",
  "content": [
    {
      "type": "text",
      "text": "Describe this image:"
    },
    {
      "type": "image_url",
      "image_url": {
        "url": "data:image/png;base64,iVBORw0KG\u2026"
      }
    }
  ]
}


### 3c. Default resolution (max_tokens, temperature)

`_resolve_defaults` looks in this order: caller kwargs → model
metadata → `ProviderSettings.default_temperature`. Standard models
send both fields directly.

In [10]:
body_default = adapter._build_request_body([Message(role="user", content="hi")])
body_override = adapter._build_request_body(
    [Message(role="user", content="hi")],
    max_tokens=512,
    temperature=0.0,
)
print(
    f"default:  max_tokens={body_default['max_tokens']}, temperature={body_default['temperature']}"
)
print(
    f"override: max_tokens={body_override['max_tokens']}, temperature={body_override['temperature']}"
)
print(f"\nstandard model has 'max_completion_tokens'? {'max_completion_tokens' in body_default}")
print(f"standard model has 'reasoning_effort'?      {'reasoning_effort' in body_default}")

default:  max_tokens=32768, temperature=0.7
override: max_tokens=512, temperature=0.0

standard model has 'max_completion_tokens'? False
standard model has 'reasoning_effort'?      False


## 4. o-series models — `_is_reasoning_model` switching

OpenAI's o-series reasoning models (`o3`, `o3-mini`, `o4-mini`,
`o1`) reject the chat-completions defaults. Two specific things
change, both keyed off `ModelMetadata.supports_thinking`:

1. **Role rewriting** — `role="system"` is rejected; the adapter
   rewrites it to `role="developer"` inside `_format_messages`.
2. **Body keys** — `max_tokens` and `temperature` are not allowed.
   The adapter sends `max_completion_tokens` and `reasoning_effort`
   instead.

This is the entire reason the field is called `supports_thinking`
rather than `is_o_series` — Anthropic's thinking models also use it,
and the adapter behavior across providers becomes a single capability
predicate rather than a per-provider name regex.

In [11]:
# Bind the same fixture config to a reasoning model
reasoning_adapter = OpenaiAdapter(fake_config, REASONING_MODEL)
print(f"model_name           = {reasoning_adapter.model_name}")
print(f"_is_reasoning_model  = {reasoning_adapter._is_reasoning_model}")
print(f"supports_thinking    = {reasoning_adapter._model_meta.supports_thinking}")

model_name           = o4-mini
_is_reasoning_model  = True
supports_thinking    = True


### 4a. system → developer role rewriting

Watch the same `Message(role="system", …)` produce different wire
shapes depending on which model the adapter is bound to.

In [12]:
messages = [
    Message(role="system", content="You are precise. Reason carefully."),
    Message(role="user", content="What is 12 * 13?"),
]

std_body = adapter._build_request_body(messages)
rsn_body = reasoning_adapter._build_request_body(messages)

print("standard model — role stays 'system':")
for m in std_body["messages"]:
    print(f"  {m['role']:9s} -> {str(m.get('content'))[:40]}")

print("\nreasoning model — role rewritten to 'developer':")
for m in rsn_body["messages"]:
    print(f"  {m['role']:9s} -> {str(m.get('content'))[:40]}")

standard model — role stays 'system':
  system    -> You are precise. Reason carefully.
  user      -> What is 12 * 13?

reasoning model — role rewritten to 'developer':
  developer -> You are precise. Reason carefully.
  user      -> What is 12 * 13?


### 4b. Body keys: `max_completion_tokens` and `reasoning_effort`

On reasoning models `max_tokens` and `temperature` are dropped.
`max_completion_tokens` is set from the resolved budget;
`reasoning_effort` defaults to `"medium"` and is overridable via
kwargs (`"low"` / `"medium"` / `"high"`).

In [13]:
rsn_default = reasoning_adapter._build_request_body(
    [Message(role="user", content="Solve this puzzle.")],
)
rsn_high = reasoning_adapter._build_request_body(
    [Message(role="user", content="Solve this puzzle.")],
    max_tokens=4096,
    reasoning_effort="high",
)

for label, body in [("default", rsn_default), ("effort=high", rsn_high)]:
    print(f"{label}:")
    for k in ("max_tokens", "temperature", "max_completion_tokens", "reasoning_effort"):
        print(f"  {k:25s} = {body.get(k)!r}")
    print()

default:
  max_tokens                = None
  temperature               = None
  max_completion_tokens     = 100000
  reasoning_effort          = 'medium'

effort=high:
  max_tokens                = None
  temperature               = None
  max_completion_tokens     = 4096
  reasoning_effort          = 'high'



### 4c. The capability matrix in the packaged catalog

OpenAI's packaged TOML at `arcllm/providers/openai.toml` is the
source of truth for which models trigger the reasoning path. Note
the GPT-4.1 family (1M-token context) is the current default, with
GPT-4o still available, and three o-series entries flagged as
thinking models.

In [14]:
real_config = load_provider_config("openai")
print(f"default model:        {real_config.provider.default_model}")
print()
print(f"{'model':18s}  thinking  context     out")
print("-" * 56)
for name, meta in real_config.models.items():
    print(
        f"{name:18s}  {meta.supports_thinking!s:8s}  {meta.context_window:>10}  {meta.max_output_tokens}"
    )

default model:        gpt-4.1

model               thinking  context     out
--------------------------------------------------------
gpt-4.1             False        1000000  32768
gpt-4.1-mini        False        1000000  32768
gpt-4.1-nano        False        1000000  32768
gpt-4o              False         128000  16384
gpt-4o-mini         False         128000  16384
o3                  True          200000  100000
o3-mini             True          200000  100000
o4-mini             True          200000  100000


In [15]:
# Gating helper: pick a sensible reasoning_effort only when the bound
# model supports it. Avoids hard-coding model name regexes downstream.
def call_kwargs(
    provider_config: ProviderConfig, model: str, *, deep: bool = False
) -> dict[str, Any]:
    meta = provider_config.models.get(model)
    if meta and meta.supports_thinking:
        return {"reasoning_effort": "high" if deep else "medium"}
    return {"temperature": 0.0}


for m in ("gpt-4.1", "gpt-4o", "o3", "o4-mini"):
    print(f"  {m:10s} -> {call_kwargs(real_config, m, deep=True)}")

  gpt-4.1    -> {'temperature': 0.0}
  gpt-4o     -> {'temperature': 0.0}
  o3         -> {'reasoning_effort': 'high'}
  o4-mini    -> {'reasoning_effort': 'high'}


## 5. Tool calls

Three things differ from Anthropic for tools:

1. **Tool definition shape** — wrapped in `{type: "function", function: {…}}`.
   The JSON Schema key stays `parameters` (Anthropic renames it to `input_schema`).
2. **Tool calls in the response live at the message level** —
   `choices[0].message.tool_calls`, not as content blocks.
3. **Arguments are JSON strings** on both legs of the round-trip.
   The adapter calls `json.dumps` outbound and `json.loads` inbound.

In [16]:
tool = Tool(
    name="search_database",
    description="Search the knowledge base",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"},
            "limit": {"type": "integer", "default": 10},
        },
        "required": ["query"],
    },
)
print(json.dumps(adapter._format_tool(tool), indent=2))

{
  "type": "function",
  "function": {
    "name": "search_database",
    "description": "Search the knowledge base",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "Search query"
        },
        "limit": {
          "type": "integer",
          "default": 10
        }
      },
      "required": [
        "query"
      ]
    }
  }
}


### 5a. Outbound: assistant turn with tool calls

When the agent loop appends the model's previous turn back to history,
the adapter has to convert ArcLLM's content-block `ToolUseBlock`s
into OpenAI's message-level `tool_calls` array. Note the
`json.dumps(arguments)` — OpenAI needs a JSON string here, not a dict.

In [17]:
msg = Message(
    role="assistant",
    content=[
        TextBlock(text="Let me look that up."),
        ToolUseBlock(id="call_01", name="search", arguments={"query": "cats"}),
        ToolUseBlock(id="call_02", name="lookup", arguments={"id": 42}),
    ],
)
formatted = adapter._format_message(msg)
print(json.dumps(formatted, indent=2))

{
  "role": "assistant",
  "content": "Let me look that up.",
  "tool_calls": [
    {
      "id": "call_01",
      "type": "function",
      "function": {
        "name": "search",
        "arguments": "{\"query\": \"cats\"}"
      }
    },
    {
      "id": "call_02",
      "type": "function",
      "function": {
        "name": "lookup",
        "arguments": "{\"id\": 42}"
      }
    }
  ]
}


In [18]:
# Tool calls without any text — content becomes None (not empty string)
msg = Message(
    role="assistant",
    content=[ToolUseBlock(id="call_01", name="calc", arguments={"x": 1})],
)
formatted = adapter._format_message(msg)
print("content is None:", formatted["content"] is None)
print(json.dumps(formatted, indent=2))

content is None: True
{
  "role": "assistant",
  "content": null,
  "tool_calls": [
    {
      "id": "call_01",
      "type": "function",
      "function": {
        "name": "calc",
        "arguments": "{\"x\": 1}"
      }
    }
  ]
}


### 5b. Tool result flattening — 1 ArcLLM message → N OpenAI messages

This is the most surprising OpenAI quirk. ArcLLM lets you put
multiple `ToolResultBlock`s in a single `Message(role="tool", …)` —
you ran two tools in parallel, here are both answers. OpenAI's API
requires one message per tool result, each with its own
`tool_call_id`. So the flattening lives at the *messages* level
(`_format_messages`), not at the per-message level (`_format_message`).

```
ArcLLM (1 message):              OpenAI (2 messages):
Message(role="tool",       -->   {"role": "tool", "tool_call_id": "t1", "content": "42"}
  content=[                -->   {"role": "tool", "tool_call_id": "t2", "content": "hello"}
    ToolResult(id="t1"),
    ToolResult(id="t2"),
  ])
```

In [19]:
messages = [
    Message(role="user", content="Do two things."),
    Message(
        role="tool",
        content=[
            ToolResultBlock(tool_use_id="call_01", content="Result from tool 1"),
            ToolResultBlock(tool_use_id="call_02", content="Result from tool 2"),
        ],
    ),
]

formatted = adapter._format_messages(messages)
print(f"input messages : {len(messages)}")
print(f"output messages: {len(formatted)}")
for m in formatted:
    print(" ", m)

input messages : 2
output messages: 3
  {'role': 'user', 'content': 'Do two things.'}
  {'role': 'tool', 'tool_call_id': 'call_01', 'content': 'Result from tool 1'}
  {'role': 'tool', 'tool_call_id': 'call_02', 'content': 'Result from tool 2'}


In [20]:
# A tool result whose content is a list of TextBlocks (not a plain string)
# is concatenated. ToolResultBlock with non-string content is the
# Anthropic-shaped path; OpenAI flattens it to a single string.
msg = Message(
    role="tool",
    content=[
        ToolResultBlock(
            tool_use_id="call_01",
            content=[TextBlock(text="row 1"), TextBlock(text="row 2")],
        ),
    ],
)
out = adapter._format_messages([msg])
print(out)

[{'role': 'tool', 'tool_call_id': 'call_01', 'content': 'row 1 row 2'}]


### 5c. Inbound: parsing `message.tool_calls`

When OpenAI replies with a tool call, the parser walks
`choices[0].message.tool_calls`, decodes each `function.arguments`
from its JSON string, and produces a list of `ToolCall` objects on
`LLMResponse.tool_calls`. The parser is defensive about the
arguments shape (string is normal, dict is occasionally seen,
anything else raises).

In [21]:
# Helper to mint canned chat-completions responses
def make_openai_text_response(
    text: str = "Hello!",
    *,
    model: str = STANDARD_MODEL,
    prompt_tokens: int = 10,
    completion_tokens: int = 5,
    finish_reason: str = "stop",
    **extra_usage: Any,
) -> dict[str, Any]:
    usage = {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
    }
    usage.update(extra_usage)
    return {
        "id": "chatcmpl-walkthrough",
        "object": "chat.completion",
        "model": model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": finish_reason,
            }
        ],
        "usage": usage,
    }


def make_openai_tool_response(
    *,
    tool_id: str = "call_01",
    tool_name: str = "search",
    tool_args: dict[str, Any] | None = None,
    text: str | None = None,
    model: str = STANDARD_MODEL,
) -> dict[str, Any]:
    return {
        "id": "chatcmpl-walkthrough",
        "object": "chat.completion",
        "model": model,
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": text,
                    "tool_calls": [
                        {
                            "id": tool_id,
                            "type": "function",
                            "function": {
                                "name": tool_name,
                                "arguments": json.dumps(tool_args or {"query": "test"}),
                            },
                        }
                    ],
                },
                "finish_reason": "tool_calls",
            }
        ],
        "usage": {"prompt_tokens": 20, "completion_tokens": 15, "total_tokens": 35},
    }


print("helpers ready.")

helpers ready.


In [22]:
# Tool-only response → ToolCall objects with parsed dict arguments
raw = make_openai_tool_response(
    tool_id="call_abc",
    tool_name="search",
    tool_args={"query": "LLM agents", "limit": 5},
)
resp = adapter._parse_response(raw)
print(f"content     = {resp.content!r}")
print(f"stop_reason = {resp.stop_reason}")
for tc in resp.tool_calls:
    print(f"  tool: {tc.name} id={tc.id} args={tc.arguments}")

content     = None
stop_reason = tool_use
  tool: search id=call_abc args={'query': 'LLM agents', 'limit': 5}


In [23]:
# Edge cases: dict input pass-through + bad JSON raises ArcLLMParseError
print("dict args pass through:")
tc = adapter._parse_tool_call(
    {
        "id": "call_1",
        "type": "function",
        "function": {"name": "calc", "arguments": {"expression": "2+2"}},
    }
)
print(f"  {tc}")

print("\nbad JSON string raises:")
try:
    adapter._parse_tool_call(
        {
            "id": "call_1",
            "type": "function",
            "function": {"name": "calc", "arguments": "not valid json {{{"},
        }
    )
except ArcLLMParseError as e:
    print(f"  caught: {type(e).__name__}, raw_string={e.raw_string!r}")

print("\nunexpected type raises:")
try:
    adapter._parse_tool_call(
        {
            "id": "call_1",
            "type": "function",
            "function": {"name": "calc", "arguments": 12345},
        }
    )
except ArcLLMParseError as e:
    print(f"  caught: {type(e).__name__}")

dict args pass through:
  id='call_1' name='calc' arguments={'expression': '2+2'}

bad JSON string raises:
  caught: ArcLLMParseError, raw_string='not valid json {{{'

unexpected type raises:
  caught: ArcLLMParseError


### 5d. Driving a two-turn loop with a mocked HTTP client

Patch `httpx.AsyncClient.post` to return turn 1 (the model decides
to call `get_weather`), then turn 2 (the model gives a final answer).
Between the turns, the harness runs the tool and appends the
assistant + tool messages to history. The adapter handles all the
translation; the harness sees only the universal types.

In [24]:
def fake_post(json_payload: dict[str, Any], status: int = 200) -> AsyncMock:
    response = httpx.Response(
        status,
        json=json_payload if status == 200 else None,
        text=json.dumps(json_payload) if status != 200 else None,
        request=httpx.Request("POST", "https://api.openai.com/v1/chat/completions"),
    )
    return AsyncMock(return_value=response)


get_weather = Tool(
    name="get_weather",
    description="Get current weather for a city",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "units": {"type": "string", "enum": ["c", "f"]},
        },
        "required": ["city"],
    },
)


async def run_two_turn_loop() -> None:
    adapter = OpenaiAdapter(fake_config, STANDARD_MODEL)

    turn1 = make_openai_tool_response(
        tool_id="call_weather_01",
        tool_name="get_weather",
        tool_args={"city": "Austin", "units": "f"},
    )
    turn2 = make_openai_text_response(text="Austin is 75 °F and sunny.")

    with patch.object(adapter._client, "post", fake_post(turn1)):
        history: list[Message] = [
            Message(role="user", content="What's the weather in Austin?"),
        ]
        first = await adapter.invoke(history, tools=[get_weather])
        print(f"turn 1: stop_reason={first.stop_reason}, tool={first.tool_calls[0].name}")

    tc = first.tool_calls[0]
    history.append(
        Message(
            role="assistant",
            content=[ToolUseBlock(id=tc.id, name=tc.name, arguments=tc.arguments)],
        )
    )
    history.append(
        Message(
            role="tool",
            content=[ToolResultBlock(tool_use_id=tc.id, content="75F sunny")],
        )
    )

    with patch.object(adapter._client, "post", fake_post(turn2)):
        final = await adapter.invoke(history, tools=[get_weather])
        print(f"turn 2: stop_reason={final.stop_reason}, content={final.content!r}")

    await adapter.close()


await run_two_turn_loop()

turn 1: stop_reason=tool_use, tool=get_weather
turn 2: stop_reason=end_turn, content='Austin is 75 °F and sunny.'


## 6. Streaming events on the wire

`OpenaiAdapter.invoke` returns a single `LLMResponse` after the full
body is in. ArcLLM does not currently expose chunk-by-chunk streaming
through the adapter API — accumulation happens at the provider layer.

Here are the SSE chunk shapes OpenAI sends when `stream=true` is set
on the request, so you know what's happening underneath. Tool-call
deltas are particularly fiddly: each chunk carries an `index` and a
partial slice of the JSON `arguments` string. A folder accumulates
them by index until the final chunk delivers `finish_reason`.

In [25]:
# OpenAI Chat Completions SSE delta shapes — illustrative.
# These are what arrive on the wire when stream=true; we do not stream
# in invoke() today.
streaming_text_chunks = [
    {
        "choices": [
            {"index": 0, "delta": {"role": "assistant", "content": ""}, "finish_reason": None}
        ]
    },
    {"choices": [{"index": 0, "delta": {"content": "Austin"}, "finish_reason": None}]},
    {"choices": [{"index": 0, "delta": {"content": " is 75"}, "finish_reason": None}]},
    {"choices": [{"index": 0, "delta": {"content": " °F."}, "finish_reason": None}]},
    {"choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]},
]
for c in streaming_text_chunks:
    print(c)

{'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': ''}, 'finish_reason': None}]}
{'choices': [{'index': 0, 'delta': {'content': 'Austin'}, 'finish_reason': None}]}
{'choices': [{'index': 0, 'delta': {'content': ' is 75'}, 'finish_reason': None}]}
{'choices': [{'index': 0, 'delta': {'content': ' °F.'}, 'finish_reason': None}]}
{'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]}


In [26]:
# Folding deltas back into the same shape _parse_response consumes.
# Even though we don't stream today, the parser is already the right
# shape — we only need to assemble the body.
def fold_text_chunks(chunks: list[dict[str, Any]]) -> dict[str, Any]:
    parts: list[str] = []
    finish_reason = "stop"
    for c in chunks:
        choice = c["choices"][0]
        delta = choice.get("delta", {})
        if delta.get("content"):
            parts.append(delta["content"])
        if choice.get("finish_reason"):
            finish_reason = choice["finish_reason"]
    return {
        "id": "chatcmpl-synth",
        "object": "chat.completion",
        "model": STANDARD_MODEL,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": "".join(parts)},
                "finish_reason": finish_reason,
            }
        ],
        "usage": {
            "prompt_tokens": 12,
            "completion_tokens": len(parts),
            "total_tokens": 12 + len(parts),
        },
    }


synth = fold_text_chunks(streaming_text_chunks)
resp = adapter._parse_response(synth)
print(f"folded content    = {resp.content!r}")
print(f"folded stop_reason= {resp.stop_reason}")
print(f"folded usage      = {resp.usage}")

folded content    = 'Austin is 75 °F.'
folded stop_reason= end_turn
folded usage      = input_tokens=12 output_tokens=3 total_tokens=15 cache_read_tokens=None cache_write_tokens=None reasoning_tokens=None


In [27]:
# Tool-call streaming: each chunk carries a partial 'arguments' string,
# keyed by tool-call index. Folder concatenates by index, then json.loads
# at the end.
streaming_tool_chunks = [
    {
        "choices": [
            {
                "index": 0,
                "delta": {
                    "role": "assistant",
                    "content": None,
                    "tool_calls": [
                        {
                            "index": 0,
                            "id": "call_str_1",
                            "type": "function",
                            "function": {"name": "search", "arguments": ""},
                        }
                    ],
                },
                "finish_reason": None,
            }
        ]
    },
    {
        "choices": [
            {
                "index": 0,
                "delta": {"tool_calls": [{"index": 0, "function": {"arguments": '{"que'}}]},
                "finish_reason": None,
            }
        ]
    },
    {
        "choices": [
            {
                "index": 0,
                "delta": {"tool_calls": [{"index": 0, "function": {"arguments": 'ry": "cats"}'}}]},
                "finish_reason": None,
            }
        ]
    },
    {"choices": [{"index": 0, "delta": {}, "finish_reason": "tool_calls"}]},
]

# Tool-arg folder by tool-call index
tools_by_index: dict[int, dict[str, Any]] = {}
for c in streaming_tool_chunks:
    delta = c["choices"][0].get("delta", {})
    for tc in delta.get("tool_calls") or []:
        idx = tc["index"]
        slot = tools_by_index.setdefault(
            idx, {"id": "", "type": "function", "function": {"name": "", "arguments": ""}}
        )
        if tc.get("id"):
            slot["id"] = tc["id"]
        fn_delta = tc.get("function") or {}
        if fn_delta.get("name"):
            slot["function"]["name"] = fn_delta["name"]
        if fn_delta.get("arguments"):
            slot["function"]["arguments"] += fn_delta["arguments"]

for idx, tc in tools_by_index.items():
    print(
        f"tool[{idx}] id={tc['id']} name={tc['function']['name']} args={tc['function']['arguments']!r}"
    )

# Reuse _parse_tool_call — it handles the JSON string for us.
parsed = adapter._parse_tool_call(tools_by_index[0])
print(f"parsed -> {parsed}")

tool[0] id=call_str_1 name=search args='{"query": "cats"}'
parsed -> id='call_str_1' name='search' arguments={'query': 'cats'}


## 7. Errors

When the API returns a non-200, `invoke()` raises `ArcLLMAPIError`
with `status_code`, `body`, `provider`, and an optional `retry_after`
parsed from the `Retry-After` response header. This is the contract
the retry / circuit-breaker / fallback modules rely on for smart
decisions.

In [28]:
async def call_with_status(
    status: int,
    body_text: str = "{}",
    retry_after: str | None = None,
) -> ArcLLMAPIError:
    adapter = OpenaiAdapter(fake_config, STANDARD_MODEL)
    headers = {"retry-after": retry_after} if retry_after else {}
    response = httpx.Response(
        status,
        text=body_text,
        headers=headers,
        request=httpx.Request("POST", "https://api.openai.com/v1/chat/completions"),
    )
    with patch.object(adapter._client, "post", AsyncMock(return_value=response)):
        try:
            await adapter.invoke([Message(role="user", content="hi")])
        except ArcLLMAPIError as e:
            return e
        finally:
            await adapter.close()
    raise AssertionError("expected ArcLLMAPIError")


# 401 — auth failure (don't retry)
err = await call_with_status(401, '{"error":{"message":"Incorrect API key"}}')
print(f"401 → status_code={err.status_code}, retry_after={err.retry_after}")
print(f"     provider     ={err.provider}")

401 → status_code=401, retry_after=None
     provider     =openai


In [29]:
# 429 — rate limited, with Retry-After header parsed as seconds
err = await call_with_status(
    429,
    '{"error":{"type":"requests","message":"Rate limit reached."}}',
    retry_after="12",
)
print(f"429 → retry_after={err.retry_after} seconds")

# 500 — provider-side failure (retry with backoff)
err = await call_with_status(500, '{"error":"oops"}')
print(f"500 → status_code={err.status_code}")

429 → retry_after=12.0 seconds
500 → status_code=500


In [30]:
# Smart retry decision — the retry module uses status_code to decide.
def should_retry(err: ArcLLMAPIError) -> str:
    if err.status_code == 429:
        return f"retry after {err.retry_after}s"
    if err.status_code >= 500:
        return "retry with backoff"
    if err.status_code in (401, 403):
        return "do not retry — credential issue"
    if err.status_code == 400:
        return "do not retry — caller bug"
    return "do not retry"


for code_, body in [
    (429, '{"error":"rate"}'),
    (500, '{"error":"oops"}'),
    (401, '{"error":"key"}'),
    (400, '{"error":"bad"}'),
]:
    e = ArcLLMAPIError(
        status_code=code_,
        body=body,
        provider="openai",
        retry_after=15.0 if code_ == 429 else None,
    )
    print(f"  {code_} → {should_retry(e)}")

  429 → retry after 15.0s
  500 → retry with backoff
  401 → do not retry — credential issue
  400 → do not retry — caller bug


### Stop-reason mapping

OpenAI's `finish_reason` values are mapped through `_STOP_REASON_MAP`
to ArcLLM's canonical `StopReason` literal — `"stop"` becomes
`"end_turn"`, `"tool_calls"` becomes `"tool_use"`, `"length"` becomes
`"max_tokens"`. The unknown-value branch falls back to `"end_turn"`
so loops don't crash on a future provider quirk.

In [31]:
from arcllm.adapters.openai import _STOP_REASON_MAP

for openai_val, arc_val in _STOP_REASON_MAP.items():
    print(f"  finish_reason={openai_val!r:18s} -> stop_reason={arc_val!r}")

print()
for raw in ("stop", "tool_calls", "length", "content_filter", "future_unknown"):
    print(f"  {raw:18s} -> {adapter._map_stop_reason(raw)!r}")

  finish_reason='stop'             -> stop_reason='end_turn'
  finish_reason='tool_calls'       -> stop_reason='tool_use'
  finish_reason='length'           -> stop_reason='max_tokens'
  finish_reason='content_filter'   -> stop_reason='content_filter'

  stop               -> 'end_turn'
  tool_calls         -> 'tool_use'
  length             -> 'max_tokens'
  content_filter     -> 'content_filter'
  future_unknown     -> 'end_turn'


### Usage parsing — including reasoning tokens

OpenAI's chat-completions `usage` block uses `prompt_tokens` and
`completion_tokens` (Anthropic uses `input_tokens` / `output_tokens`).
On o-series responses, `completion_tokens_details.reasoning_tokens`
carries the hidden chain-of-thought token count — ArcLLM lifts that
to `Usage.reasoning_tokens` so cost-tracking modules can attribute
spend correctly.

In [32]:
raw = make_openai_text_response(prompt_tokens=200, completion_tokens=150)
raw["usage"]["completion_tokens_details"] = {"reasoning_tokens": 80}

resp = adapter._parse_response(raw)
print(resp.usage)

input_tokens=200 output_tokens=150 total_tokens=350 cache_read_tokens=None cache_write_tokens=None reasoning_tokens=80


## 8. (live) one real call gated on OPENAI_API_KEY

A single live call against the real OpenAI API, gated on
`HAS_LIVE_KEY`. If you have `OPENAI_API_KEY` exported, this runs
end-to-end through the real adapter, the real config, and a real
network request. Otherwise it cleanly skips.

In [33]:
if not HAS_LIVE_KEY:
    print("skip — set OPENAI_API_KEY")
    raise SystemExit

real_config = load_provider_config("openai")
real_model = real_config.provider.default_model


async def live_call() -> None:
    async with OpenaiAdapter(real_config, real_model) as adapter:
        try:
            resp = await adapter.invoke(
                [Message(role="user", content="Reply with exactly the word: OK")],
                max_tokens=10,
                temperature=0.0,
            )
        except ArcLLMAPIError as e:
            # Auth/quota/rate-limit failures shouldn't crash the smoke run.
            # The same error surface we exercised under mocks in §7 fires here.
            print(f"live call failed: {type(e).__name__} status={e.status_code}")
            return
        print(f"model         = {resp.model}")
        print(f"content       = {resp.content!r}")
        print(f"stop_reason   = {resp.stop_reason}")
        print(f"input_tokens  = {resp.usage.input_tokens}")
        print(f"output_tokens = {resp.usage.output_tokens}")


await live_call()

live call failed: ArcLLMAPIError status=429


## 9. Summary

`OpenaiAdapter` is a small, sharply-scoped class that translates
between ArcLLM's universal types and the OpenAI Chat Completions
API. Every method has one job; every quirk lives in exactly one
place — most notably the `_is_reasoning_model` predicate that
switches between the standard and o-series wire shapes.

What we covered:

- **Adapter contract** — `invoke`, `validate_config`, `close`. Bearer
  auth at init via `BaseAdapter`. Headers cached because the API key
  is immutable for the adapter's lifetime.
- **Standard models** — `gpt-4.1`, `gpt-4.1-mini`, `gpt-4o`, `gpt-4o-mini`.
  System messages stay in-line; `max_tokens` and `temperature` flow
  through unchanged.
- **o-series reasoning models** — `o3`, `o3-mini`, `o4-mini` keyed off
  `ModelMetadata.supports_thinking`. `_format_messages` rewrites
  `role="system"` to `role="developer"`. `_build_request_body`
  swaps `max_tokens`/`temperature` for `max_completion_tokens`/
  `reasoning_effort`. The packaged catalog at
  `arcllm/providers/openai.toml` is the source of truth.
- **Tool calls** — outbound `ToolUseBlock` becomes `tool_calls` with
  JSON-string arguments via `json.dumps`. Inbound `_parse_tool_call`
  decodes via `_parse_arguments`, accepting dict pass-through and
  raising `ArcLLMParseError` on garbage.
- **Tool result flattening** — one `Message(role="tool", content=[N blocks])`
  expands to N OpenAI messages, each with its own `tool_call_id`.
  Lives in `_format_messages`, not `_format_message` — clean
  separation of concerns.
- **Streaming** — illustrative SSE chunk shapes for both text and
  tool-call deltas, plus a folder showing how to assemble them into
  the same body shape `_parse_response` already consumes.
- **Errors** — `ArcLLMAPIError(status_code, body, provider, retry_after)`
  carries enough information for the retry / circuit-breaker /
  fallback modules to make smart decisions. `_STOP_REASON_MAP`
  normalizes `finish_reason` values; unknown falls back to
  `"end_turn"`.
- **Usage** — `prompt_tokens`/`completion_tokens` map to
  `Usage.input_tokens`/`output_tokens`;
  `completion_tokens_details.reasoning_tokens` lifts to
  `Usage.reasoning_tokens` on o-series responses.

API surface exercised in this notebook:

- `OpenaiAdapter` — public class, all private methods walked
- `BaseAdapter` — inherited; `validate_config`, `close`,
  `_parse_arguments`, `_resolve_defaults`, `_parse_retry_after`
- `ProviderConfig`, `ProviderSettings`, `ModelMetadata` — fixture and
  real config
- `Message`, `TextBlock`, `ImageBlock`, `ToolUseBlock`,
  `ToolResultBlock`, `Tool` — every input shape
- `LLMResponse`, `ToolCall`, `Usage`, `StopReason` — every output field
- `ArcLLMAPIError`, `ArcLLMParseError`, `ArcLLMConfigError` — error
  contract
- `load_provider_config` — packaged OpenAI catalog

**Next:** see `arcllm/13-open-providers` for the full adapter catalog
and capability matrix; `arcllm/03-anthropic-adapter` for the
Anthropic-side equivalent; `arcllm/07-module-system` for how the
modules (retry, fallback, circuit_breaker, queue) consume
`ArcLLMAPIError`.